# 02 · Retrieval và Generation

Biến câu hỏi thành truy vấn có bộ lọc, tìm bằng dense, BM25 và hybrid, rerank, rồi sinh câu trả lời có trích dẫn.

**Đầu ra cho người sau:** `src/rag.py`: `search_dense`, `search_bm25`, `rrf`, `ask_rag`.

**Cầu nối sang Người 3:** ở P2-11 câu 3, đáp án đúng từ pandas groupby nằm ngay cạnh nhưng RAG không tính ra. Cần cấu trúc quan hệ và khả năng tính trên toàn tập.

**Cell tuỳ chọn:** P2-08 (rerank) bỏ được nếu máy chậm; slide 13 vẫn nói được bằng sơ đồ phễu.

In [13]:
%run ./00_setup.ipynb

### P2-01 · Dùng lại chỉ mục và mô hình của Người 1, không embed lại
**ẨN**

Mục tiêu: kiểm tra chỉ mục Qdrant của Người 1 và nạp mô hình embedding từ cache.

In [14]:
# P2-01 · ẨN
df = pd.read_parquet(CLEAN_PATH)
assert qdrant_store.count() > 0, "chạy notebook 01 trước để nạp Qdrant"
_ = vector.embed_query("khởi động mô hình")

### P2-02 · LLM trích bộ lọc, pydantic kiểm tra và đổi 5 tỷ thành 5.000 triệu
**CACHE**

Mục tiêu: hiểu câu hỏi: LLM trích bộ lọc thành JSON, pydantic kiểm tra và chuẩn hoá đơn vị.

In [15]:
# P2-02 · CACHE
assert rag.parse_money_mil("5 tỷ") == 5000 and rag.parse_money_mil("1 tỷ 2") == 1200
asks = ["Căn hộ 2PN ở Thủ Đức dưới 5 tỷ",
        "Nhà Cầu Giấy giá từ 5 tỷ đến 8,5 tỷ, diện tích trên 40m2",
        "Đất ở Đà Nẵng tầm 1 tỷ 2 trở xuống"]
rows = []
for q in asks:
    f, out = rag.parse_filters(q)  # gọi Claude lần đầu, sau đó đọc cache
    rows.append({"câu hỏi": q, "LLM trích": {k: v for k, v in out.data.items() if v is not None},
                 "bộ lọc (giá: triệu đồng)": f.model_dump(exclude_none=True)})
pd.DataFrame(rows)

,câu hỏi,LLM trích,bộ lọc (giá: triệu đồng)
0,Căn hộ 2PN ở Thủ Đức dưới 5 tỷ,"{'district': 'Thủ Đức', 'property_type': 'Căn hộ chung cư', 'bedrooms': 2, '...","{'district': 'Thủ Đức', 'property_type': 'Căn hộ chung cư', 'bedrooms': 2, '..."
1,"Nhà Cầu Giấy giá từ 5 tỷ đến 8,5 tỷ, diện tích trên 40m2","{'district': 'Cầu Giấy', 'property_type': 'Nhà', 'min_price': '5 tỷ', 'max_p...","{'district': 'Cầu Giấy', 'property_type': 'Nhà', 'min_price_mil': 5000.0, 'm..."
2,Đất ở Đà Nẵng tầm 1 tỷ 2 trở xuống,"{'province': 'Đà Nẵng', 'property_type': 'Đất', 'max_price': '1.2 tỷ'}","{'province': 'Đà Nẵng', 'property_type': 'Đất', 'max_price_mil': 1200.0}"


### P2-03 · Cosine đo hướng, không đo độ dài: (6, 8) vẫn được 1,00
**LIVE**

Mục tiêu: tính cosine bằng numpy, đối chiếu sklearn: 1,00 / 0,96 / 0,00 cho q = (3, 4).

In [16]:
# P2-03 · LIVE
from sklearn.metrics.pairwise import cosine_similarity
q = (3, 4)
for d in [(6, 8), (4, 3), (-4, 3)]:
    mine, ref = rag.cosine(q, d), cosine_similarity([q], [d])[0, 0]
    assert abs(mine - ref) < 1e-12
    print(d, f"numpy {mine:.2f} | sklearn {ref:.2f}")
assert [round(rag.cosine(q, d), 2) for d in [(6, 8), (4, 3), (-4, 3)]] == [1.00, 0.96, 0.00]

(6, 8) numpy 1.00 | sklearn 1.00
(4, 3) numpy 0.96 | sklearn 0.96
(-4, 3) numpy 0.00 | sklearn 0.00


### P2-04 · Ngưỡng điểm cắt bớt tin kém liên quan khi tăng k
**CACHE**

Mục tiêu: dense search top-k kèm ngưỡng; thử k = 3, 5, 8.

In [17]:
# P2-04 · CACHE
q, threshold = "Căn hộ 2PN ở Thủ Đức giá bao nhiêu?", 0.80
dense_k = cached("p2_04_dense.pkl", lambda: {k: search_dense(q, k=k, threshold=threshold) for k in (3, 5, 8)})
for k, hits in dense_k.items():
    print(f"k = {k}: {len(hits)} tin có cosine ≥ {threshold}")
rag.hits_table(dense_k[8])

k = 3: 3 tin có cosine ≥ 0.8
k = 5: 5 tin có cosine ≥ 0.8
k = 8: 8 tin có cosine ≥ 0.8


,listing_id,điểm,giá,m²,địa chỉ,đoạn
0,15405,0.9147,"5,70 tỷ",61.0,Hồ Chí Minh,Bán căn hộ 02 PN 61m2 New City Thủ Thiêm TP Thủ Đức · Căn hộ chung cư
1,3138,0.9118,"3,15 tỷ",73.4,"Thủ Đức, Hồ Chí Minh","Bán căn 2PN 2WC 73,4m2 MT Eastmark City giá chỉ 3,15 tỷ bao sang tên."
2,18456,0.9099,"1,95 tỷ",63.0,"Tân Phú, Thủ Đức, Hồ Chí Minh","Bán căn 3PN CC Topaz Home 2 - Thủ Đức · Căn hộ chung cư · Tân Phú, Thủ"
3,14515,0.9097,"3,20 tỷ",69.3,"Linh Xuân, Thủ Đức, Hồ Chí Minh","Căn hộ 2PN 2WC, 69.3m² tại TP. Thủ Đức Chỉ từ 3,2 tỷ. Ở chuẩn Nhật, ch"
4,749,0.9095,"4,60 tỷ",70.0,"Thủ Đức, Hồ Chí Minh","Bán căn hộ 2PN + 2WC Lumière Masteri Center Point 4,6 tỷ full nội thất"
5,378,0.9076,"17,00 tỷ",94.0,"Thủ Thiêm, Thủ Đức, Hồ Chí Minh","Duy nhất trong tháng 6 mới có giá ưu đãi này, bán nhanh căn 2PN lầu ca"
6,1198,0.9046,"10,00 tỷ",88.0,"An Lợi Đông, Thủ Đức, Hồ Chí Minh",CĂN HỘ CAO CẤP 2pn tại KHU ĐÔ THỊ SALA - View thành phố 88m2. Chỉ 10 t
7,10432,0.9043,"2,05 tỷ",74.0,"Thảo Điền, Thủ Đức, Hồ Chí Minh","Bán nhanh căn hộ 2PN 2WC, 74m2, Masteri An Phú, Thảo Điền, Q2, giá 2,0"


### P2-05 · BM25 khớp đúng từ khoá: 5/5 tin có cả sổ hồng và 2PN, dense chỉ 3/5
**CACHE**

Mục tiêu: BM25 (sparse vector trên Qdrant) với bộ tách từ tiếng Việt; so sánh "sổ hồng 2PN" và "gần trường học" với dense.

In [18]:
# P2-05 · CACHE
pairs = {"sổ hồng 2PN": ["sổ hồng", "2pn"], "gần trường học": ["gần trường học"]}
both = cached("p2_05_bm25_dense.pkl",
              lambda: {q: {"BM25": search_bm25(q, 5), "dense": search_dense(q, 5)} for q in pairs})
print("tách từ:", rag.tokenize_vi("sổ hồng 2PN gần trường học"))
rows = []
for q, terms in pairs.items():
    for method, hits in both[q].items():
        exact = hits["text"].str.lower().map(lambda t: all(x in t for x in terms)).sum()
        rows.append({"câu": q, "cách": method, "top-5 chứa nguyên cụm": f"{exact}/5",
                     "listing_id": hits["listing_id"].tolist()})
pd.DataFrame(rows)

tách từ: ['sổ', 'hồng', '2pn', 'gần', 'trường_học']


,câu,cách,top-5 chứa nguyên cụm,listing_id
0,sổ hồng 2PN,BM25,5/5,"[7661, 4009, 15659, 4033, 12986]"
1,sổ hồng 2PN,dense,3/5,"[8951, 10362, 4251, 16349, 17868]"
2,gần trường học,BM25,3/5,"[10327, 634, 13342, 4694, 16398]"
3,gần trường học,dense,2/5,"[13446, 8179, 6169, 14157, 541]"


### P2-06 · Hai bảng xếp hạng gộp thành một bằng RRF
**LIVE**

Mục tiêu: chạy toy X, Y, Z (assert) rồi chạy câu thật.

In [19]:
# P2-06 · LIVE
toy = rrf([["X", "Y", "Z"], ["Y", "Z", "X"]])
assert [d for d, _ in toy] == ["Y", "X", "Z"]
print([(d, f"{s:.5f}") for d, s in toy])

q = "Căn hộ 2PN ở Thủ Đức có sổ hồng"
dense, bm25 = search_dense(q, 20), search_bm25(q, 20)
rank = lambda hits, i: hits.listing_id.tolist().index(i) + 1 if i in set(hits.listing_id) else None
pd.DataFrame([{"listing_id": i, "hạng dense": rank(dense, i), "hạng BM25": rank(bm25, i), "RRF": round(s, 5)}
              for i, s in rrf([dense.listing_id.tolist(), bm25.listing_id.tolist()])[:5]])

[('Y', '0.03252'), ('X', '0.03227'), ('Z', '0.03200')]


,listing_id,hạng dense,hạng BM25,RRF
0,4033,4,7.0,0.03055
1,3138,3,14.0,0.02939
2,17868,6,11.0,0.02924
3,13089,17,19.0,0.02565
4,14515,1,NaN,0.01639


### P2-07 · Pre-filter giữ đủ k tin đúng điều kiện, post-filter làm rơi gần hết
**LIVE**

Mục tiêu: pre-filter và post-filter trên cùng một câu hỏi có điều kiện cứng.

In [20]:
# P2-07 · LIVE
q = "Căn hộ 2PN ở Thủ Đức dưới 3 tỷ"
f = rag.QueryFilters(district="Thủ Đức", property_type="Căn hộ chung cư", bedrooms=2, max_price_mil=3000)
k = 10
pre = search_dense(q, k=k, filters=f)                    # lọc trong vector DB rồi lấy top-k
post = rag.post_filter(search_dense(q, k=k), f)          # lấy top-k rồi mới lọc
print(f"pre-filter: {len(pre)}/{k} tin, tất cả thoả điều kiện")
print(f"post-filter: {len(post)}/{k} tin còn lại sau khi lọc")
rag.hits_table(pre).head(5)

pre-filter: 10/10 tin, tất cả thoả điều kiện
post-filter: 2/10 tin còn lại sau khi lọc


,listing_id,điểm,giá,m²,địa chỉ,đoạn
0,165,0.9127,"2,90 tỷ",56.00,"Cát Lái, Thủ Đức, Hồ Chí Minh",Đừng để căn hộ dưới 3 tỷ tại thành phố Hồ Chí Minh chỉ còn là kỷ niệm
1,14485,0.9089,"2,40 tỷ",71.00,"Thủ Đức, Hồ Chí Minh",Bán gấp screc tower quận 2 71m2 2pn chỉ 2 tỷ 400 triệu · Căn hộ chung
2,1132,0.9071,"2,60 tỷ",59.00,"Long Thạnh Mỹ, Thủ Đức, Hồ Chí Minh",Giá chỉ từ 2.6 tỷ - bán gấp các căn hộ 1PN + 1 & 2PN - Tặng ngay gói t
3,651,0.9061,"1,90 tỷ",55.00,"Thạnh Mỹ Lợi, Thủ Đức, Hồ Chí Minh",Suất cuối cùng của Block C view sông giá chủ đầu tư - anh chị nhanh ta
4,7388,0.9012,"2,01 tỷ",62.13,"Thạnh Mỹ Lợi, Thủ Đức, Hồ Chí Minh",Chung cư xã hội xây như thương mại chỉ 33 triệu/ m2. Mở bán block C cu


### P2-08 · Rerank xếp lại thứ tự: tin hạng 24 theo RRF lên hạng 1
**CACHE**

Mục tiêu: rerank cross-encoder: top-50 xuống top-5 (cell tuỳ chọn, bỏ nếu máy chậm).

In [21]:
# P2-08 · CACHE
q = "Căn hộ gần trường học, có hồ bơi"
cands = rag.search_hybrid(q, k=50, candidates=50)
top = cached("p2_08_rerank.pkl", lambda: rag.rerank(q, cands, top=5))
before = {lid: i + 1 for i, lid in enumerate(cands.listing_id)}
pd.DataFrame({"listing_id": top.listing_id, "hạng trước (RRF)": top.listing_id.map(before),
              "hạng sau (rerank)": range(1, len(top) + 1), "điểm rerank": top.rerank.round(3),
              "đoạn": top.text.str.slice(0, 70)})

,listing_id,hạng trước (RRF),hạng sau (rerank),điểm rerank,đoạn
0,1170,24,1,2.162,"Bán căn hộ sân vườn 1PN, 80m2 - 2 ban công giá tốt tại Orchard Hill\nCă"
1,16446,33,2,1.162,"Căn Bcons Solary 2PN view hồ bơi tầng trung đẹp. Tặng Ti vi, vay 0% lã"
2,3683,1,3,1.041,"Bán CHCC Kingdom 101, Tô Hiến Thành, Q10. 73m2=2PN=2WC giá 5.9tỷ. LH:"
3,2228,26,4,0.820,"Bán căn hộ chung cư tại Chánh Hưng Giai Việt, 4,65 tỷ, 115m2 chủ nhà t"
4,14139,28,5,0.797,"Cần bán căn hộ chung cư The Avila, Q.8, 4PN, tặng 1 số nội thất, Phù h"


### P2-09 · Câu trả lời bám nguồn, mỗi ý có [Tin#ID]
**LIVE**

Mục tiêu: ghép prompt (chỉ thị, ngữ cảnh, câu hỏi) rồi sinh câu trả lời kèm [Tin#ID].

In [22]:
# P2-09 · LIVE
q = "Căn hộ 2PN ở Thủ Đức giá bao nhiêu?"
found = rag.retrieve(q)
contexts = rag.format_context(found["hits"])
prompt = rag.build_prompt(q, contexts)
print("CHỈ THỊ:", rag.ANSWER_SYSTEM[:160], "…")
print("PROMPT (rút gọn):", prompt[:400].replace("\n", " "), "…\n")
answer = rag.generate(q, contexts)
print("TRẢ LỜI:\n" + answer.text)
print("\nnguồn được trích:", rag.cited_ids(answer.text), "| nguồn đã đưa vào:", found["hits"].listing_id.tolist())

CHỈ THỊ: Bạn là trợ lý thị trường bất động sản Việt Nam. Chỉ trả lời từ NGỮ CẢNH được cung cấp.
Mỗi nhận định phải kèm nguồn dạng [Tin#ID]. Nếu ngữ cảnh không đủ để trả  …
PROMPT (rút gọn): NGỮ CẢNH: <tai_lieu> [Tin#1198] CĂN HỘ CAO CẤP 2pn tại KHU ĐÔ THỊ SALA - View thành phố 88m2. Chỉ 10 tỷ bao gồm THUẾ PHÍ Căn hộ chung cư · An Lợi Đông, Thủ Đức, Hồ Chí Minh · giá 10.00 tỷ · 88 m² · 2 phòng ngủ Căn hộ cao cấp khu đô thị Sala - Sadora Mai Chí Thọ, Thủ Thiêm, Thủ Đức 2PN - 88m2 View thành phố Giá bán: 10.[phone_number] VNĐ - đã bao gồm thuế và phí Mọi chi tiết vui lòng liên hệ sđt [p …

TRẢ LỜI:
- 10 tỷ cho căn hộ 2PN, 88 m² tại Khu đô thị Sala – Sadora, Thủ Thiêm, Thủ Đức【Tin#1198】  
- 3,2 tỷ cho căn hộ 2PN, 69,3 m² tại Linh Xuân, Thủ Đức【Tin#14515】  
- 3,22 tỷ cho căn hộ 2PN, 70 m² tại Linh Xuân, Thủ Đức (giá 46 triệu / m²)【Tin#8940】【Tin#12955】  
- 4,9 tỷ cho căn hộ 2PN, 74,53 m² tại Lumiere Boulevard, Long Bình, Quận 9 (thuộc Thủ Đức)【Tin#8972】  

nguồn được trích: [1198, 14515, 894

### P2-10 · Đóng gói thành một hàm ask_rag cho Người 4
**ẨN**

Mục tiêu: ask_rag(question) trả về câu trả lời và nguồn (định nghĩa trong src/rag.py).

In [23]:
# P2-10 · ẨN
_r = ask_rag("Căn hộ 2PN ở Thủ Đức giá bao nhiêu?")
assert {"answer", "sources", "contexts", "latency", "input_tokens", "output_tokens"} <= set(_r)

### P2-11 · RAG trả lời đúng tra cứu và ràng buộc nhưng sai câu tổng hợp
**LIVE**

Mục tiêu: 3 câu: tra cứu (đúng), có ràng buộc (đúng), tổng hợp (sai có chủ đích); in đáp án pandas groupby.

In [24]:
# P2-11 · LIVE
three = {"tra cứu": "Nhà 6 tầng có thang máy ở Ngọa Long, Bắc Từ Liêm giá bao nhiêu?",
         "ràng buộc": "Căn hộ chung cư 2 phòng ngủ ở Thủ Đức giá dưới 3 tỷ có căn nào?",
         "tổng hợp": "Ở Hồ Chí Minh, quận nào có giá/m² trung bình thấp nhất (chỉ tính quận có ít nhất 20 tin)?"}
for kind, q in three.items():
    r = ask_rag(q)
    print(f"[{kind}] {q}\n→ {r['answer'][:300]}\n  nguồn: {r['sources']}\n")

g = (df[df.province == "Hồ Chí Minh"].dropna(subset=["district", "price_m2_mil"])
     .groupby("district")["price_m2_mil"].agg(["mean", "count"]).query("count >= 20"))
print(f"ĐÁP ÁN pandas groupby: {g['mean'].idxmin()} ({g['mean'].min():.2f} triệu/m², trên {len(g)} quận)")

[tra cứu] Nhà 6 tầng có thang máy ở Ngọa Long, Bắc Từ Liêm giá bao nhiêu?
→ Nhà 6 tầng có thang máy tại Ngõ Ngoại Long, Bắc Từ Liêm có giá **7,58 tỷ** đồng. [Tin#18590]
  nguồn: [18590, 11250, 1301, 10135, 1191]

[ràng buộc] Căn hộ chung cư 2 phòng ngủ ở Thủ Đức giá dưới 3 tỷ có căn nào?
→ Có 5 dự án căn hộ 2 phòng ngủ ở Thủ Đức có giá dưới 3 tỷ:

| Dự án | Vị trí (Quận) | Diện tích | Giá | Nguồn |
|------|---------------|-----------|-----|------|
| Citi Grand | Cát Lái, Thủ Đức | 56 m² | 2.90 tỷ | [Tin#165] |
| Thủ Thiêm Green House (Block C) | Thạnh Mỹ Lợi, Thủ Đức | 55 m² | 1.90 tỷ
  nguồn: [165, 651, 1132, 7388, 19943]

[tổng hợp] Ở Hồ Chí Minh, quận nào có giá/m² trung bình thấp nhất (chỉ tính quận có ít nhất 20 tin)?
→ Không đủ dữ liệu để xác định quận nào ở Hồ Chí Minh có giá/m² trung bình thấp nhất (chỉ tính các quận có ít nhất 20 tin). [Tin#15022] [Tin#6020] [Tin#11782] [Tin#17359] [Tin#14826]
  nguồn: [15022, 6020, 11782, 17359, 14826]

ĐÁP ÁN pandas groupby: Củ Chi (19.98 tr